In [0]:
%sh
ls /dbfs/mnt/bd/openuniverselake/ais2021/delta


In [0]:

%sh
ls /dbfs/mnt/bd/openuniverselake/aisraw/aisdata/

In [0]:
CREATE OR REPLACE TEMP VIEW ais2021
AS SELECT * FROM delta.`/mnt/bd/openuniverselake/ais2021/delta`; 

In [0]:
%python
columns = [f"{col}" for col in spark.table("ais2021").columns]
select_stmt = f"SELECT {', '.join(columns)} FROM ais2021"
print(select_stmt)

In [0]:
SELECT COUNT(*), count(distinct *) FROM ais2021;

In [0]:
CREATE OR REPLACE TEMPORARY VIEW ais_csv
USING csv
OPTIONS (
  path '/mnt/bd/openuniverselake/aisraw/aisdata/csv',
  header 'true'
);

In [0]:
CREATE OR REPLACE TEMPORARY VIEW ais_raw 
AS
SELECT rec_time, mobile_type, mmsi, latitude, longitude, nav_status, rot, sog, cog, heading, imo, callsign, name, ship_type, cargo_type, ship_width, ship_length, device, draught, destination, eta, source_type, a, b, c, d FROM ais2021
UNION ALL
SELECT `# Timestamp` rec_time, `Type of mobile` mobile_type, MMSI  mmsi, Latitude latitude, Longitude longitude, `Navigational status` nav_status, ROT rot, SOG sog, COG cog, Heading heading, IMO imo, Callsign callsign, Name name
     , `Ship type` ship_type, `Cargo type` cargo_type, Width ship_width, Length ship_length, `Type of position fixing device` device, Draught draught, Destination destination, ETA eta, `Data source type` source_type, A a, B b, C c,  D d 
FROM ais_csv;

In [0]:
CREATE OR REPLACE TEMPORARY VIEW ais
AS
SELECT to_timestamp(rec_time, 'dd/MM/yyyy HH:mm:ss') as rec_time
     , mobile_type, mmsi
     , cast(latitude as NUMERIC(9,6)) AS latitude , CAST( longitude as NUMERIC(9,6)) AS longitude
     , nav_status
     , CAST( rot as double) AS rot
     , CAST( sog as double) AS sog
     , CAST( cog as double) AS cog
     , CAST( heading as double) AS heading
     , imo, callsign, name, ship_type, cargo_type
     , CAST( ship_width as double) AS ship_width
     , CAST( ship_length as double) AS ship_length
     , device
     , CAST( draught as double) AS draught
     , destination,  eta, source_type
     , CAST( a as double) AS a
     , CAST( b as double) AS b
     , CAST( c as double) AS c
     , CAST( d as double) AS d 
     , year(to_timestamp(rec_time, 'dd/MM/yyyy HH:mm:ss')) as yyyy
     , date_format(to_timestamp(rec_time, 'dd/MM/yyyy HH:mm:ss'), 'yyyy-MM') as yyyy_mm
     , date_format(to_timestamp(rec_time, 'dd/MM/yyyy HH:mm:ss'), 'yyyy-MM-dd') as yyyy_mm_dd
FROM ais_raw;


In [0]:
%python
df = spark.sql("select * from ais")
df.coalesce(1).write.partitionBy("yyyy", "yyyy_mm", "yyyy_mm_dd").option("maxRecordsPerFile", 20000000).format("delta").mode("overwrite").save("/mnt/bd/openuniverselake/aisraw/aisdata/delta")

In [0]:
SELECT * FROM delta.`/mnt/bd/openuniverselake/aisraw/aisdata/delta` LIMIT 10;